&nbsp;

## GENCODE promoter windows (GENCODE release 48)

- (2026-09-08) 後続の解析である `fanta.bio`における promoter の証拠としても使用できるため、統合する
- (2026-09-08) `fanta.bio`の BEDファイルと後で `bedtools intersect` しておくので、BEDファイルを作成しておく
- → `gffread` では、どうやら [gencode.v48.promoter_windows.gff3.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/gencode.v48.promoter_windows.gff3.gz) の feature の タイプ名のところが Sequence Ontology Feature Annotation (SOFA) に該当しない？ ため、うまく変換できないので、データフレームとして操作して変換する
- [../02_fantabio/02_5_ensembl_promoter_intersect.ipynb](../02_fantabio/02_5_ensembl_promoter_intersect.ipynb)に処理を移動
- https://www.gencodegenes.org/pages/promoter_windows/
- [gencode.v48.promoter_windows.gff3.gz](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/gencode.v48.promoter_windows.gff3.gz)

In [1]:
import os
import datetime
import pathlib
# import numpy as np
# import pandas as pd
import polars as pl
import plotly.express as px
from IPython.display import display

pl.Config.set_tbl_rows(6)
pl.Config.set_fmt_str_lengths(500)
os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
# https://www.ncbi.nlm.nih.gov/datasets/docs/v2/reference-docs/file-formats/annotation-files/about-ncbi-gff3/
# Caution: The information is based on GENCODE v48.
gencode_promoter_windows = pl.read_csv(
    "../../Data/Database/GENCODE_v49/metadata/gencode.v48.promoter_windows.gff3",
    separator="\t",
    comment_prefix="#",
    has_header=False,
    n_threads=16,
    new_columns=[
        "chromosome",
        "source",
        "feature",
        "start",
        "end",
        "score",
        "strand",
        "phase",
        "attribute"
    ]
)

display(gencode_promoter_windows)

chromosome,source,feature,start,end,score,strand,phase,attribute
str,str,str,i64,i64,str,str,str,str
"""chr1""","""GENCODE""","""promoter_window""",64419,65418,""".""","""+""",""".""","""ID=ENSG00000186092_promoter_window;Name=OR4F5_promoter_window;gene_id=ENSG00000186092.7;transcript_id=ENST00000641515.2;promoter_window_type=MANE_Select_with_TSS;window_size=1000"""
"""chr1""","""GENCODE""","""promoter_window""",451679,452678,""".""","""-""",""".""","""ID=ENSG00000284733_promoter_window;Name=OR4F29_promoter_window;gene_id=ENSG00000284733.2;transcript_id=ENST00000426406.4;promoter_window_type=MANE_Select_without_TSS;window_size=1000"""
"""chr1""","""GENCODE""","""promoter_window""",686655,687654,""".""","""-""",""".""","""ID=ENSG00000284662_promoter_window;Name=OR4F16_promoter_window;gene_id=ENSG00000284662.2;transcript_id=ENST00000332831.5;promoter_window_type=MANE_Select_without_TSS;window_size=1000"""
…,…,…,…,…,…,…,…,…
"""KI270734.1""","""GENCODE""","""promoter_window""",71411,72410,""".""","""+""",""".""","""ID=ENSG00000276017_promoter_window;Name=ENSG00000276017_promoter_window;gene_id=ENSG00000276017.1;transcript_id=ENST00000617983.1;promoter_window_type=Non_MANE_Select_model;window_size=1000"""
"""KI270734.1""","""GENCODE""","""promoter_window""",130494,131493,""".""","""+""",""".""","""ID=ENSG00000278817_promoter_window;Name=ENSG00000278817_promoter_window;gene_id=ENSG00000278817.1;transcript_id=ENST00000613204.1;promoter_window_type=Non_MANE_Select_model;window_size=1000"""
"""KI270734.1""","""GENCODE""","""promoter_window""",161853,162852,""".""","""-""",""".""","""ID=ENSG00000277196_promoter_window;Name=ENSG00000277196_promoter_window;gene_id=ENSG00000277196.4;transcript_id=ENST00000621424.4;promoter_window_type=Non_MANE_Select_model;window_size=1000"""


In [3]:
gencode_promoter_windows_parsed = gencode_promoter_windows.with_columns([
    pl.col("attribute").str.extract(r"ID=([^;]+)", group_index=1).alias("ID"),
    pl.col("attribute").str.extract(r"Name=([^;]+)", group_index=1).alias("Name"),
    pl.col("attribute").str.extract(r"gene_id=([^;]+)", group_index=1).alias("gene_id"),
    pl.col("attribute").str.extract(r"transcript_id=([^;]+)", group_index=1).alias("transcript_id"),
    pl.col("attribute").str.extract(r"promoter_window_type=([^;]+)", group_index=1).alias("promoter_window_type"),
    pl.col("attribute").str.extract(r"window_size=([^;]+)", group_index=1).alias("window_size"),
    pl.col("attribute").str.extract(r"ensembl_promoter_id=([^;]+)", group_index=1).alias("ensembl_promoter_id"),
]).drop(
    "attribute"
).sort(
    by="gene_id",
    descending=False
)

display(gencode_promoter_windows_parsed)

chromosome,source,feature,start,end,score,strand,phase,ID,Name,gene_id,transcript_id,promoter_window_type,window_size,ensembl_promoter_id
str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str
"""chrX""","""GENCODE""","""promoter_window""",100636807,100637806,""".""","""-""",""".""","""ENSG00000000003_promoter_window""","""TSPAN6_promoter_window""","""ENSG00000000003.16""","""ENST00000373020.9""","""MANE_Select_with_TSS""","""1000""","""ENSRX_93GXHH"""
"""chrX""","""GENCODE""","""promoter_window""",100583936,100584935,""".""","""+""",""".""","""ENSG00000000005_promoter_window""","""TNMD_promoter_window""","""ENSG00000000005.6""","""ENST00000373031.5""","""MANE_Select_with_TSS""","""1000""",null
"""chr20""","""GENCODE""","""promoter_window""",50958533,50959532,""".""","""-""",""".""","""ENSG00000000419_promoter_window""","""DPM1_promoter_window""","""ENSG00000000419.14""","""ENST00000371588.10""","""MANE_Select_with_TSS""","""1000""","""ENSR20_9Q84Z"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr14""","""GENCODE""","""promoter_window""",61320954,61321953,""".""","""+""",""".""","""ENSG00000310561_promoter_window""","""ENSG00000310561_promoter_window""","""ENSG00000310561.1""","""ENST00000850887.1""","""Non_MANE_Select_model""","""1000""","""ENSR14_BG6KQ"""
"""chrX""","""GENCODE""","""promoter_window""",45729995,45730994,""".""","""+""",""".""","""ENSG00000310562_promoter_window""","""ENSG00000310562_promoter_window""","""ENSG00000310562.1""","""ENST00000850889.1""","""Non_MANE_Select_model""","""1000""",null
"""chr19""","""GENCODE""","""promoter_window""",1274530,1275529,""".""","""+""",""".""","""ENSG00000310576_promoter_window""","""FAM174C_promoter_window""","""ENSG00000310576.1""","""ENST00000409293.6""","""MANE_Select_with_TSS""","""1000""","""ENSR19_B8SJ"""


In [4]:
gencode_promoter_windows_transcript = gencode_promoter_windows_parsed.select(
    pl.col("transcript_id"),
    pl.col("ID"),
    # pl.col("gene_id"),
    pl.col("promoter_window_type"),
    # pl.col("window_size"),
    pl.col("ensembl_promoter_id")
).unique(
    subset=["transcript_id", "promoter_window_type", "ensembl_promoter_id"]
)

display(gencode_promoter_windows_transcript)

transcript_id,ID,promoter_window_type,ensembl_promoter_id
str,str,str,str
"""ENST00000369304.8""","""ENSG00000143127_promoter_window""","""MANE_Select_with_TSS""","""ENSR1_945WND"""
"""ENST00000221538.8""","""ENSG00000104941_promoter_window""","""MANE_Select_with_TSS""","""ENSR19_BC4HD"""
"""ENST00000243349.13""","""ENSG00000123612_promoter_window""","""MANE_Select_with_TSS""","""ENSR2_B369M5"""
…,…,…,…
"""ENST00000366690.5""","""ENSG00000168118_promoter_window""","""MANE_Select_with_TSS""","""ENSR1_7HH263"""
"""ENST00000395297.6""","""ENSG00000147724_promoter_window""","""MANE_Select_with_TSS""","""ENSR8_7B5TXR"""
"""ENST00000380833.9""","""ENSG00000073464_promoter_window""","""MANE_Select_with_TSS""","""ENSRX_7L5DN"""


In [5]:
# gffread ではうまく変換できないので、データフレームとして操作してBEDファイル (BED6 format)として保存しておく
gencode_promoter_windows_bed = gencode_promoter_windows_parsed.select(
    pl.col("chromosome"),
    (pl.col("start") - 1).alias("start"),  # GFF3 1-based inclusive -> BED 0-based half-open
    pl.col("end"),
    pl.col("ID"),
    pl.col("score"),
    pl.col("strand"),
    # pl.col("transcript_id"),
    # pl.col("promoter_window_type"),
    # pl.col("window_size"),
    # pl.col("ensembl_promoter_id").fill_null("."),
)

# gencode_promoter_windows_bed.write_csv(
#     "../../Data/Database/GENCODE_v49/metadata/gencode.v48.promoter_windows_v260908.bed",
#     include_header=False,
#     separator="\t",
#     null_value=".",
#     quote_style="never",
# )

display(gencode_promoter_windows_bed)

chromosome,start,end,ID,score,strand
str,i64,i64,str,str,str
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""",""".""","""-"""
"""chrX""",100583935,100584935,"""ENSG00000000005_promoter_window""",""".""","""+"""
"""chr20""",50958532,50959532,"""ENSG00000000419_promoter_window""",""".""","""-"""
…,…,…,…,…,…
"""chr14""",61320953,61321953,"""ENSG00000310561_promoter_window""",""".""","""+"""
"""chrX""",45729994,45730994,"""ENSG00000310562_promoter_window""",""".""","""+"""
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""",""".""","""+"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## `bedtools intersect`

- 実行コマンド
- `cwltool --outdir ./out/fantabio/bedtools_intersect/ ./Tools/bedtools_intersect_v2.cwl ./config/bedtools_intersect/gencode_v48_promoter_windows_intersect_fantabio_cre.yml`

In [6]:
fantabio_cre_intersect_gencode_v48_promoter_windows_bed = pl.read_csv(
    "../../out/fantabio/bedtools_intersect/gencode.v48.promoter_windows_v260908_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    has_header=False,
    n_threads=16,
    new_columns=[
        "chromosome_ensembl",
        "start_ensembl",
        "end_ensembl",
        "ensembl_promoter_accession",
        "score",
        "strand",
        "chromosome_fantabio",
        "start_fantabio",
        "end_fantabio",
        "cre_id",
        "score_cre",
        "strand_cre",
        "thick_start_cre",
        "thick_end_cre",
        "item_rgb",
        "cre_name",
        "pooled_cre_scores",
        "intersect_length"
    ]
).with_columns(
    # 念のため確認
    whether_promoter_window_is_contained_in_cre=pl.col("chromosome_ensembl") == pl.col("chromosome_fantabio")
).filter(
    pl.col("whether_promoter_window_is_contained_in_cre") == True
).drop(
    "whether_promoter_window_is_contained_in_cre"
).select(
    "chromosome_ensembl",
    "start_ensembl",
    "end_ensembl",
    "ensembl_promoter_accession",
    "strand",
    "chromosome_fantabio",
    "start_fantabio",
    "end_fantabio",
    "cre_id",
    "cre_name",
    "pooled_cre_scores",
    "intersect_length"
).with_columns(
    (pl.col("intersect_length") / (pl.col("end_ensembl") - pl.col("start_ensembl"))).alias("fantabio_cre_overlap_frac_of_ensembl_promoter_window"),
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
).drop(
    "pooled_cre_scores"
)

display(fantabio_cre_intersect_gencode_v48_promoter_windows_bed)

chromosome_ensembl,start_ensembl,end_ensembl,ensembl_promoter_accession,strand,chromosome_fantabio,start_fantabio,end_fantabio,cre_id,cre_name,intersect_length,fantabio_cre_overlap_frac_of_ensembl_promoter_window,directionality,fantabio_cre_class
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""-""","""chrX""",100636532,100637120,"""FCHS_442038""","""cp4@TSPAN6""",314,0.314,-0.998632,"""PLA"""
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""-""","""chrX""",100637146,100637301,"""FCHS_442039""","""cp3@TSPAN6""",155,0.155,-1.0,"""ELA"""
"""chrX""",100583935,100584935,"""ENSG00000000005_promoter_window""","""+""","""chrX""",100584757,100585110,"""FCHS_592490""","""cp6@TNMD""",178,0.178,1.0,"""PLA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr14""",61320953,61321953,"""ENSG00000310561_promoter_window""","""+""","""chr14""",61321845,61322211,"""FCHS_124311""","""cp26@PRKCH""",108,0.108,-0.333333,"""ELA"""
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""+""","""chr19""",1274940,1275250,"""FCHS_196118""","""FCHS_196118""",310,0.31,0.908046,"""ELA"""
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""+""","""chr19""",1275363,1275715,"""FCHS_196119""","""cp3@FAM174C""",166,0.166,0.991043,"""PLA"""


In [7]:
gencode_promoter_windows_parse_for_join = gencode_promoter_windows_parsed.select(
    "chromosome",
    "start",
    "end",
    "ID",
    "window_size",
    "strand",
    "ensembl_promoter_id",
    "Name",
    "gene_id",
    "transcript_id",
    "promoter_window_type"
).with_columns(
    (pl.col("start") - 1).alias("start"), # GFF3 1-based inclusive -> BED 0-based half-open
).rename(
    {
        "chromosome": "chromosome_ensembl",
        "start": "start_ensembl",
        "end": "end_ensembl",
        "ID": "ensembl_promoter_accession",
        "window_size": "ensembl_promoter_window_size"
    }
)

display(gencode_promoter_windows_parse_for_join)

chromosome_ensembl,start_ensembl,end_ensembl,ensembl_promoter_accession,ensembl_promoter_window_size,strand,ensembl_promoter_id,Name,gene_id,transcript_id,promoter_window_type
str,i64,i64,str,str,str,str,str,str,str,str
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""1000""","""-""","""ENSRX_93GXHH""","""TSPAN6_promoter_window""","""ENSG00000000003.16""","""ENST00000373020.9""","""MANE_Select_with_TSS"""
"""chrX""",100583935,100584935,"""ENSG00000000005_promoter_window""","""1000""","""+""",null,"""TNMD_promoter_window""","""ENSG00000000005.6""","""ENST00000373031.5""","""MANE_Select_with_TSS"""
"""chr20""",50958532,50959532,"""ENSG00000000419_promoter_window""","""1000""","""-""","""ENSR20_9Q84Z""","""DPM1_promoter_window""","""ENSG00000000419.14""","""ENST00000371588.10""","""MANE_Select_with_TSS"""
…,…,…,…,…,…,…,…,…,…,…
"""chr14""",61320953,61321953,"""ENSG00000310561_promoter_window""","""1000""","""+""","""ENSR14_BG6KQ""","""ENSG00000310561_promoter_window""","""ENSG00000310561.1""","""ENST00000850887.1""","""Non_MANE_Select_model"""
"""chrX""",45729994,45730994,"""ENSG00000310562_promoter_window""","""1000""","""+""",null,"""ENSG00000310562_promoter_window""","""ENSG00000310562.1""","""ENST00000850889.1""","""Non_MANE_Select_model"""
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""1000""","""+""","""ENSR19_B8SJ""","""FAM174C_promoter_window""","""ENSG00000310576.1""","""ENST00000409293.6""","""MANE_Select_with_TSS"""


In [8]:
fantabio_cre_intersect_gencode_v48_promoter_windows_bed_join = fantabio_cre_intersect_gencode_v48_promoter_windows_bed.join(
    gencode_promoter_windows_parse_for_join,
    on=[
        "chromosome_ensembl",
        "start_ensembl",
        "end_ensembl",
        "ensembl_promoter_accession",
        "strand"
    ],
    how="inner",
    coalesce=True
)

display(fantabio_cre_intersect_gencode_v48_promoter_windows_bed_join)

chromosome_ensembl,start_ensembl,end_ensembl,ensembl_promoter_accession,strand,chromosome_fantabio,start_fantabio,end_fantabio,cre_id,cre_name,intersect_length,fantabio_cre_overlap_frac_of_ensembl_promoter_window,directionality,fantabio_cre_class,ensembl_promoter_window_size,ensembl_promoter_id,Name,gene_id,transcript_id,promoter_window_type
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str,str,str,str,str,str,str
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""-""","""chrX""",100636532,100637120,"""FCHS_442038""","""cp4@TSPAN6""",314,0.314,-0.998632,"""PLA""","""1000""","""ENSRX_93GXHH""","""TSPAN6_promoter_window""","""ENSG00000000003.16""","""ENST00000373020.9""","""MANE_Select_with_TSS"""
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""-""","""chrX""",100637146,100637301,"""FCHS_442039""","""cp3@TSPAN6""",155,0.155,-1.0,"""ELA""","""1000""","""ENSRX_93GXHH""","""TSPAN6_promoter_window""","""ENSG00000000003.16""","""ENST00000373020.9""","""MANE_Select_with_TSS"""
"""chrX""",100583935,100584935,"""ENSG00000000005_promoter_window""","""+""","""chrX""",100584757,100585110,"""FCHS_592490""","""cp6@TNMD""",178,0.178,1.0,"""PLA""","""1000""",null,"""TNMD_promoter_window""","""ENSG00000000005.6""","""ENST00000373031.5""","""MANE_Select_with_TSS"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr14""",61320953,61321953,"""ENSG00000310561_promoter_window""","""+""","""chr14""",61321845,61322211,"""FCHS_124311""","""cp26@PRKCH""",108,0.108,-0.333333,"""ELA""","""1000""","""ENSR14_BG6KQ""","""ENSG00000310561_promoter_window""","""ENSG00000310561.1""","""ENST00000850887.1""","""Non_MANE_Select_model"""
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""+""","""chr19""",1274940,1275250,"""FCHS_196118""","""FCHS_196118""",310,0.31,0.908046,"""ELA""","""1000""","""ENSR19_B8SJ""","""FAM174C_promoter_window""","""ENSG00000310576.1""","""ENST00000409293.6""","""MANE_Select_with_TSS"""
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""+""","""chr19""",1275363,1275715,"""FCHS_196119""","""cp3@FAM174C""",166,0.166,0.991043,"""PLA""","""1000""","""ENSR19_B8SJ""","""FAM174C_promoter_window""","""ENSG00000310576.1""","""ENST00000409293.6""","""MANE_Select_with_TSS"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Check

- Prefer PLA, then the CRE with larger overlap of the promoter window.

In [10]:
# Prefer PLA, then the CRE with larger overlap of the promoter window.
fantabio_cre_one_per_promoter = (
    fantabio_cre_intersect_gencode_v48_promoter_windows_bed
    .sort(
        "ensembl_promoter_accession",
        pl.col("fantabio_cre_class") == "PLA",
        "fantabio_cre_overlap_frac_of_ensembl_promoter_window",
        descending=[False, True, True],
    )
    .unique(
        subset="ensembl_promoter_accession",
        keep="first",
        maintain_order=True,
    )
)

fantabio_cre_one_per_promoter.write_csv(
    "../../out/fantabio/ensembl_promoter/gencode.v48.promoter_windows_v260908_intersecting_human-CREv1.2.1.hg38.cre-peaks_v261003_one_per_promoter.tsv",
    separator="\t"
)

display(fantabio_cre_one_per_promoter)

chromosome_ensembl,start_ensembl,end_ensembl,ensembl_promoter_accession,strand,chromosome_fantabio,start_fantabio,end_fantabio,cre_id,cre_name,intersect_length,fantabio_cre_overlap_frac_of_ensembl_promoter_window,directionality,fantabio_cre_class
str,i64,i64,str,str,str,i64,i64,str,str,i64,f64,f64,str
"""chrX""",100636806,100637806,"""ENSG00000000003_promoter_window""","""-""","""chrX""",100636532,100637120,"""FCHS_442038""","""cp4@TSPAN6""",314,0.314,-0.998632,"""PLA"""
"""chrX""",100583935,100584935,"""ENSG00000000005_promoter_window""","""+""","""chrX""",100584757,100585110,"""FCHS_592490""","""cp6@TNMD""",178,0.178,1.0,"""PLA"""
"""chr20""",50958532,50959532,"""ENSG00000000419_promoter_window""","""-""","""chr20""",50958344,50959007,"""FCHS_259378""","""cp4@DPM1,cp1@MOCS3""",475,0.475,-0.864704,"""PLA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr9""",136991421,136992421,"""ENSG00000310560_promoter_window""","""+""","""chr9""",136991967,136992621,"""FCHS_433338""","""cp4@PAXX""",454,0.454,0.970768,"""PLA"""
"""chr14""",61320953,61321953,"""ENSG00000310561_promoter_window""","""+""","""chr14""",61321157,61321954,"""FCHS_124309""","""cp30@PRKCH""",796,0.796,0.91713,"""PLA"""
"""chr19""",1274529,1275529,"""ENSG00000310576_promoter_window""","""+""","""chr19""",1275363,1275715,"""FCHS_196119""","""cp3@FAM174C""",166,0.166,0.991043,"""PLA"""
